## 2026-06-09 Update Notes

- Remote disconnect handling: simulation posting now catches host reset / request disconnect errors, waits 5 minutes for the first two reconnect attempts, then waits 1 hour for later reconnect attempts.
- Formal alpha submission: use `parse_renamed_alpha_log(...)` to extract `Renaming <alpha_id> to <name>` lines, then `submit_alpha_queue(...)` to submit one by one with a local daily limit of 4 successful submissions.
- Submission log: results are written to `runs/submissions/submission_log.jsonl`, so reruns skip already successful alpha ids and keep the daily quota count.
- Field-aware runs: save field-specific outputs under paths that include the data field id. WebDataScope neutralization stats can be used to choose a per-field neutralization before simulation.
- CLI shortcut: `python scripts/submit_alpha_queue.py --today 2026-06-09 --records "alpha_id=name,..."`.


## 1, Import Library

In [1]:
import os

In [2]:
from pathlib import Path
import importlib
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
SRC_ROOT = PROJECT_ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

import consultant_core.machine_lib as machine_lib

# Jupyter 会缓存已导入模块；修改 src/consultant_core/machine_lib.py 后需要显式 reload。
machine_lib = importlib.reload(machine_lib)

check_submission = machine_lib.check_submission
first_order_factory = machine_lib.first_order_factory
get_alphas = machine_lib.get_alphas
get_datafields = machine_lib.get_datafields
get_group_second_order_factory = machine_lib.get_group_second_order_factory
load_task_pool = machine_lib.load_task_pool
login = machine_lib.login
multi_simulate = machine_lib.multi_simulate
next_simulation_start = machine_lib.next_simulation_start
parse_renamed_alpha_log = machine_lib.parse_renamed_alpha_log
process_datafields = machine_lib.process_datafields
prune = machine_lib.prune
submit_alpha_queue = machine_lib.submit_alpha_queue
trade_when_factory = machine_lib.trade_when_factory
ts_ops = machine_lib.ts_ops
view_alphas = machine_lib.view_alphas

#参数
DATASET_ID = "analyst4"

def run_simulation_stage(pools, stage_name, log_filename, dataset_id=DATASET_ID, neut="SUBINDUSTRY", region="USA", universe="TOP3000"):
    log_path = PROJECT_ROOT / "runs" / "alpha_machine" / dataset_id / log_filename
    start = next_simulation_start(log_path)
    print(f"从 pool {start} 继续{stage_name}回测，日志: {log_path}")
    multi_simulate(pools, neut, region, universe, start, log_path=log_path)
    return log_path

## 2, 登录
<div style="margin-left: 20px;">
1, 在machine_lib文件的login方法中填写用户名和密码后保存，然后来到本文件Restart Kernal后重新import machine_lib后才在本文件生效
</div>

<div style="margin-left: 20px;">
2, 打印INVALID_CREDENTAIL即登录失败，打印自己的user_id信息才是登录成功。
</div>

In [3]:
s = login()

## 3, 获取数据字段
<div style="margin-left: 20px;">
在官网Data页面中显示的为自己目前有权限的数据集，在数据集Description面板下可以看到dataset_id
</div>

In [4]:
df = get_datafields(s, dataset_id=DATASET_ID, region='USA', universe='TOP3000', delay=1)
df

,id,description,dataset,category,subcategory,region,delay,universe,type,dateCoverage,coverage,userCount,alphaCount,pyramidMultiplier,themes
0,actual_cashflow_per_share_value_quarterly,Cash Flow Per Share - actual value for the qua...,"{'id': 'analyst4', 'name': 'Analyst Estimate D...","{'id': 'analyst', 'name': 'Analyst'}","{'id': 'analyst-analyst-estimates', 'name': 'A...",USA,1,TOP3000,MATRIX,1.0,0.4684,311,440,1.2,"[{'id': 'Jypw8X4', 'name': 'USA/D1 Fast Datase..."
1,actual_dividend_value_quarterly,Dividend - Actual value for the quarter,"{'id': 'analyst4', 'name': 'Analyst Estimate D...","{'id': 'analyst', 'name': 'Analyst'}","{'id': 'analyst-analyst-estimates', 'name': 'A...",USA,1,TOP3000,MATRIX,1.0,0.6120,93,119,1.2,"[{'id': 'Jypw8X4', 'name': 'USA/D1 Fast Datase..."
2,actual_eps_value_quarterly,Earnings Per Share (Income Statement/Per Share...,"{'id': 'analyst4', 'name': 'Analyst Estimate D...","{'id': 'analyst', 'name': 'Analyst'}","{'id': 'analyst-analyst-estimates', 'name': 'A...",USA,1,TOP3000,MATRIX,1.0,1.0000,475,885,1.2,"[{'id': 'Jypw8X4', 'name': 'USA/D1 Fast Datase..."
3,actual_sales_value_annual,Sales - Actual Value,"{'id': 'analyst4', 'name': 'Analyst Estimate D...","{'id': 'analyst', 'name': 'Analyst'}","{'id': 'analyst-analyst-estimates', 'name': 'A...",USA,1,TOP3000,MATRIX,1.0,0.9943,248,330,1.2,"[{'id': 'Jypw8X4', 'name': 'USA/D1 Fast Datase..."
4,actual_sales_value_quarterly,Sales - Value in financial services income sta...,"{'id': 'analyst4', 'name': 'Analyst Estimate D...","{'id': 'analyst', 'name': 'Analyst'}","{'id': 'analyst-analyst-estimates', 'name': 'A...",USA,1,TOP3000,MATRIX,1.0,1.0000,291,478,1.2,"[{'id': 'Jypw8X4', 'name': 'USA/D1 Fast Datase..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1319,total_goodwill_avg,Average value of goodwill assets for the period.,"{'id': 'analyst4', 'name': 'Analyst Estimate D...","{'id': 'analyst', 'name': 'Analyst'}","{'id': 'analyst-analyst-estimates', 'name': 'A...",USA,1,TOP3000,MATRIX,1.0,0.4350,2,2,1.2,"[{'id': 'Jypw8X4', 'name': 'USA/D1 Fast Datase..."
1320,total_goodwill_max,Highest estimate of goodwill assets for the pe...,"{'id': 'analyst4', 'name': 'Analyst Estimate D...","{'id': 'analyst', 'name': 'Analyst'}","{'id': 'analyst-analyst-estimates', 'name': 'A...",USA,1,TOP3000,MATRIX,1.0,0.4350,4,4,1.2,"[{'id': 'Jypw8X4', 'name': 'USA/D1 Fast Datase..."
1321,total_goodwill_median,Median value of goodwill assets for the period.,"{'id': 'analyst4', 'name': 'Analyst Estimate D...","{'id': 'analyst', 'name': 'Analyst'}","{'id': 'analyst-analyst-estimates', 'name': 'A...",USA,1,TOP3000,MATRIX,1.0,0.4350,2,2,1.2,"[{'id': 'Jypw8X4', 'name': 'USA/D1 Fast Datase..."
1322,total_goodwill_min,Lowest estimate of goodwill assets for the per...,"{'id': 'analyst4', 'name': 'Analyst Estimate D...","{'id': 'analyst', 'name': 'Analyst'}","{'id': 'analyst-analyst-estimates', 'name': 'A...",USA,1,TOP3000,MATRIX,1.0,0.4350,2,3,1.2,"[{'id': 'Jypw8X4', 'name': 'USA/D1 Fast Datase..."


## 4，数据字段预处理

<div style="margin-left: 20px;">
1, matrix, vector 数据类型
</div>

<div style="margin-left: 20px;">
2, ts_backfill 回填缺失值，提高数据Coverage 
</div>

<div style="margin-left: 20px;">
2, winsorize 去极值
</div>

In [5]:
pc_fields = process_datafields(df)
len(pc_fields)

1543

## 5, Alpha factory 
<div style="margin-left: 20px;">
在factory方法中将数据字段与操作符组装成alpha表达式
</div>

In [6]:
first_order = first_order_factory(pc_fields, ts_ops)
print(first_order[:10])
print(len(first_order))

['winsorize(ts_backfill(actual_cashflow_per_share_value_quarterly, 120), std=4)', 'ts_rank(winsorize(ts_backfill(actual_cashflow_per_share_value_quarterly, 120), std=4), 5)', 'ts_rank(winsorize(ts_backfill(actual_cashflow_per_share_value_quarterly, 120), std=4), 22)', 'ts_rank(winsorize(ts_backfill(actual_cashflow_per_share_value_quarterly, 120), std=4), 66)', 'ts_rank(winsorize(ts_backfill(actual_cashflow_per_share_value_quarterly, 120), std=4), 120)', 'ts_rank(winsorize(ts_backfill(actual_cashflow_per_share_value_quarterly, 120), std=4), 240)', 'ts_zscore(winsorize(ts_backfill(actual_cashflow_per_share_value_quarterly, 120), std=4), 5)', 'ts_zscore(winsorize(ts_backfill(actual_cashflow_per_share_value_quarterly, 120), std=4), 22)', 'ts_zscore(winsorize(ts_backfill(actual_cashflow_per_share_value_quarterly, 120), std=4), 66)', 'ts_zscore(winsorize(ts_backfill(actual_cashflow_per_share_value_quarterly, 120), std=4), 120)']
86408


## 6, 回测前载入

<div style="margin-left: 20px;">
1, alpha表达式与初始decay配对
</div>

<div style="margin-left: 20px;">
2, 保持固定顺序，便于失败后按日志续跑
</div>

<div style="margin-left: 20px;">
2, Load task pool数据结构
</div>

In [7]:
# 赋予alpha表达式一个初始decay
init_decay = 6
fo_alpha_list = []

for alpha in first_order:
    fo_alpha_list.append((alpha, init_decay))

print("数量: %s"%len(fo_alpha_list))
print(fo_alpha_list[:5])

数量: 86408
[('winsorize(ts_backfill(actual_cashflow_per_share_value_quarterly, 120), std=4)', 6), ('ts_rank(winsorize(ts_backfill(actual_cashflow_per_share_value_quarterly, 120), std=4), 5)', 6), ('ts_rank(winsorize(ts_backfill(actual_cashflow_per_share_value_quarterly, 120), std=4), 22)', 6), ('ts_rank(winsorize(ts_backfill(actual_cashflow_per_share_value_quarterly, 120), std=4), 66)', 6), ('ts_rank(winsorize(ts_backfill(actual_cashflow_per_share_value_quarterly, 120), std=4), 120)', 6)]


In [8]:
# Load alphas to task pools
fo_pools = load_task_pool(fo_alpha_list, 5, 8)
print(fo_pools[0])

[[('winsorize(ts_backfill(actual_cashflow_per_share_value_quarterly, 120), std=4)', 6), ('ts_rank(winsorize(ts_backfill(actual_cashflow_per_share_value_quarterly, 120), std=4), 5)', 6), ('ts_rank(winsorize(ts_backfill(actual_cashflow_per_share_value_quarterly, 120), std=4), 22)', 6), ('ts_rank(winsorize(ts_backfill(actual_cashflow_per_share_value_quarterly, 120), std=4), 66)', 6), ('ts_rank(winsorize(ts_backfill(actual_cashflow_per_share_value_quarterly, 120), std=4), 120)', 6)], [('ts_rank(winsorize(ts_backfill(actual_cashflow_per_share_value_quarterly, 120), std=4), 240)', 6), ('ts_zscore(winsorize(ts_backfill(actual_cashflow_per_share_value_quarterly, 120), std=4), 5)', 6), ('ts_zscore(winsorize(ts_backfill(actual_cashflow_per_share_value_quarterly, 120), std=4), 22)', 6), ('ts_zscore(winsorize(ts_backfill(actual_cashflow_per_share_value_quarterly, 120), std=4), 66)', 6), ('ts_zscore(winsorize(ts_backfill(actual_cashflow_per_share_value_quarterly, 120), std=4), 120)', 6)], [('ts_zsc

## 7, 回测

In [9]:
# Simulate First Order
fo_log = run_simulation_stage(fo_pools, "一阶", "fo_simulation_progress.jsonl")

从 pool 0 继续一阶回测，日志: d:\code\WorldQuant Brain\consultant\runs\alpha_machine\analyst4\fo_simulation_progress.jsonl
pool 0 task 7 post done
pool 0 task 7 simulate done
pool 1 task 7 post done
pool 1 task 7 simulate done
pool 2 task 7 post done
pool 2 task 7 simulate done
location key error: {"detail":"CONCURRENT_SIMULATION_LIMIT_EXCEEDED"}
pool 3 task 7 post done
pool 3 task 6 simulate done
location key error: {"detail":"CONCURRENT_SIMULATION_LIMIT_EXCEEDED"}
pool 4 task 7 post done
pool 4 task 6 simulate done
pool 5 task 7 post done
pool 5 task 7 simulate done
pool 6 task 7 post done
pool 6 task 7 simulate done
location key error: {"detail":"CONCURRENT_SIMULATION_LIMIT_EXCEEDED"}
pool 7 task 7 post done
pool 7 task 6 simulate done
location key error: {"detail":"CONCURRENT_SIMULATION_LIMIT_EXCEEDED"}
pool 8 task 7 post done
pool 8 task 6 simulate done
location key error: {"detail":"CONCURRENT_SIMULATION_LIMIT_EXCEEDED"}
pool 9 task 7 post done
pool 9 task 6 simulate done
location key erro

KeyboardInterrupt: 

## 8, 筛选Alpha


<div style="margin-left: 20px;">
1, get_alpha：截取有潜力提升表现至可以提交的alpha进入下一阶
</div>

<div style="margin-left: 20px;">
2, 剪枝Prune：精减相似alpha，提高回测资源利用率
</div>

In [ ]:
## get promising alphas to improve in the next order
fo_tracker = get_alphas("11-04", "06-06", 1.2, 0.7, "USA", 100, "track")
print(len(fo_tracker))

#### Prune 剪枝

In [ ]:
fo_layer = prune(fo_tracker, 'anl4', 5)

# 剪枝后数量
print(len(fo_layer))

## 9, 二阶提升
### ts_ops(field, days) -> group_ops(ts_ops(field, days), group)

In [ ]:
so_alpha_list = []
group_ops = ["group_neutralize", "group_rank", "group_zscore"]

for expr, decay in fo_layer:
    for alpha in get_group_second_order_factory([expr], group_ops, "USA"):
        so_alpha_list.append((alpha,decay))

print(len(so_alpha_list))
print(so_alpha_list[:3])

### Simulate second order

In [ ]:
so_pools = load_task_pool(so_alpha_list, 5, 8)
so_log = run_simulation_stage(so_pools, "二阶", "so_simulation_progress.jsonl")

## 10，三阶提升
group_ops(ts_ops(field, days), group) -> trade_when(entre_event, group_ops(ts_ops(field, days), group), exit_event)

In [ ]:
## get promising alphas from second order to improve in the third order
so_tracker = get_alphas("11-04", "06-06", 1.3, 0.8, "USA", 100, "track")

so_layer = prune(so_tracker, 'anl4', 5)
th_alpha_list = []

for expr, decay in so_layer:
    for alpha in trade_when_factory("trade_when",expr,"USA"):
        th_alpha_list.append((alpha,decay))

print("三阶表达式数量:%s"%len(th_alpha_list))

### Simulate Third Order

In [ ]:
# Simulate third order
th_pools = load_task_pool(th_alpha_list, 5, 8)
th_log = run_simulation_stage(th_pools, "三阶", "th_simulation_progress.jsonl")

## 11, 获取可提交的Alpha

<div style="margin-left: 20px;">
1, 拉取sharpe,fitness达到提交要求的alpha
</div>

<div style="margin-left: 20px;">
2, Check Submission：检查其他Test是否达到要求
</div>

<div style="margin-left: 20px;">
2, view_alphas 对可以提交的alpha进行排序
</div>


In [ ]:
# 1.58 sharpe, 1 fitness, "submit"参数
th_tracker = get_alphas("11-04", "06-06", 1.58, 1, "USA", 200, "submit")

In [ ]:
## 将get的alpha的id取出至stone_bag，用api check submission
stone_bag = []
for alpha in th_tracker:
    stone_bag.append(alpha[0])
print(len(stone_bag))
gold_bag = []
check_submission(stone_bag, gold_bag, 0)

In [ ]:
# 打印可提交的alpha信息并按sharpe排序，在网页上找到alpha手动提交
view_alphas(gold_bag)

## 12, 微调可以提交的alpha

<div style="margin-left: 20px;">
1, 得到更好的表现
</div>
<div style="margin-left: 40px;">
调整中性化，操作符参数，Decay
</div>

<div style="margin-left: 20px;">
2, Alpha质量评估
</div>

<div style="margin-left: 40px;">
performance comparison，turnover，margin
</div>

<div style="margin-left: 20px;">
3, 鲁棒性评估，防止过拟合
</div>

<div style="margin-left: 40px;">
更改中性化，Rank，Binary Test...
</div>

### Appendix

In [ ]:
# 模板构建Factory实例

def template_factory(sent_fields, option_fields):
    alpha_list = []
    for sent_field in sent_fields:
        for opt_field in option_fields:
            alpha_list.append("log(1+sigmoid(ts_zscore(%s,30))*sigmoid(ts_zscore(%s,30))"%(sent_field, opt_field))
    return alpha_list 

opt_df = get_datafields(s, dataset_id = 'option8', region='USA', universe='TOP3000', delay=1)
opt_fields = opt_df[opt_df['type'] == "MATRIX"]["id"].tolist()
print(opt_fields)

sent_df = get_datafields(s, dataset_id = 'sentiment1', region='USA', universe='TOP3000', delay=1)
sent_fields = sent_df[sent_df['type'] == "MATRIX"]["id"].tolist()
print(sent_fields)

alpha_list = template_factory(sent_fields, opt_fields)
print(alpha_list)